# Feature Engineering Thời gian & Lag (Temporal & Lag Feature Engineering)

Notebook này triển khai **temporal và lag feature engineering** cho bài toán dự báo doanh số bán lẻ Ecuador.

## Các Feature được tạo ra

### Temporal Features (Đặc trưng Thời gian)
| Feature | Mô tả |
|---|---|
| `year` | Năm trong lịch |
| `month` | Tháng trong năm (1–12) |
| `day` | Ngày trong tháng (1–31) |
| `day_of_week` | Thứ trong tuần (0 = Thứ Hai, 6 = Chủ Nhật) |
| `week_of_year` | Số tuần ISO trong năm (1–53) |
| `quarter` | Quý trong năm (1–4) |
| `is_weekend` | 1 nếu là Thứ Bảy hoặc Chủ Nhật |
| `is_month_start` | 1 nếu là ngày đầu tiên của tháng |
| `is_month_end` | 1 nếu là ngày cuối cùng của tháng |
| `is_payday` | 1 nếu là ngày 15 hoặc ngày cuối tháng (ngày phát lương ở Ecuador) |

### Lag & Rolling Features
| Feature | Mô tả |
|---|---|
| `lag_1`, `lag_2`, `lag_3` | Lag ngắn hạn (doanh số 1–3 ngày trước) |
| `lag_7` | Cùng thứ trong tuần trước |
| `lag_14`, `lag_28`, `lag_364` | Lag hai tuần, một tháng và một năm |
| `rolling_mean_7` | Rolling mean 7 ngày (shift-1 safe) |
| `rolling_mean_14` | Rolling mean 14 ngày (shift-1 safe) |
| `rolling_mean_28` | Rolling mean 28 ngày (shift-1 safe) |
| `rolling_std_7` | Rolling std 7 ngày (shift-1 safe) |

## Tổng quan Pipeline
```
Load Data → Add Temporal Features → Sort by Group + Date
    → Add Lag Features → Add Rolling Stats → Verify (No Leakage)
```

> **Đảm bảo chống Data Leakage**: tất cả lag và rolling window đều được tính với `shift(1)` để quan sát hiện tại `t` không bao giờ được đưa vào giá trị feature của chính nó.

## 1. Import Thư viện

Sử dụng **pandas** cho thao tác dữ liệu và **numpy** cho các phép tính số học.

In [14]:
import pandas as pd
import numpy as np

## 2. Đọc Dữ liệu (Load Data)

DataFrame đầu vào phải có tối thiểu các cột sau:
- **`date`** — ngày giao dịch (kiểu datetime hoặc string có thể parse được).
- **`sales`** — biến mục tiêu cần tạo lag.
- **`store_nbr`** — mã định danh store (grouping key).
- **`family`** — nhóm sản phẩm (grouping key).

In [15]:
BASE_PATH = r'D:\Topic_13_Project\Topic_13_Retail_Store_Sales_Time_Series\data'

df = pd.read_csv(rf'{BASE_PATH}\processed\train_cleaned.csv')

print(f'Shape : {df.shape}')
print(f'Columns: {df.columns.tolist()}')
df.head(3)

Shape : (3000888, 6)
Columns: ['id', 'date', 'store_nbr', 'family', 'sales', 'onpromotion']


,id,date,store_nbr,family,sales,onpromotion
0,0,2013-01-01,1,AUTOMOTIVE,0.0,0
1,1,2013-01-01,1,BABY CARE,0.0,0
2,2,2013-01-01,1,BEAUTY,0.0,0


---
## Phần 1 — Temporal Features (Đặc trưng Thời gian)

Temporal features phân rã cột `date` thành các thành phần lịch để lộ ra **tính chu kỳ** (weekly, monthly, yearly cycles) và **các ngày đặc biệt** thường gây ra đột biến doanh số.

### 3. Bước 1a — Parse Date & Trích xuất Đơn vị Lịch

Ta chuyển `date` sang `datetime` và trích xuất sáu đơn vị lịch cơ bản:
- `year`, `month`, `day` — vị trí tuyệt đối trong lịch.
- `day_of_week` — nắm bắt **quy luật nhu cầu theo tuần** (0 = Thứ Hai, 6 = Chủ Nhật).
- `week_of_year` — số tuần ISO, hữu ích cho mô hình hóa seasonality theo năm.
- `quarter` — nắm bắt **chu kỳ kinh doanh theo quý** (Q4 = mùa lễ hội).

In [16]:
df = df.copy()
df['date'] = pd.to_datetime(df['date'])

dt = df['date'].dt

df['year']         = dt.year
df['month']        = dt.month
df['day']          = dt.day
df['day_of_week']  = dt.dayofweek           # 0 = Monday, 6 = Sunday
df['week_of_year'] = dt.isocalendar().week.astype(int)
df['quarter']      = dt.quarter
df['is_weekend']   = (dt.dayofweek >= 5).astype(int)
df['is_month_start'] = dt.is_month_start.astype(int)
df['is_month_end']   = dt.is_month_end.astype(int)
df['is_payday']    = ((dt.day == 15) | dt.is_month_end).astype(int)

print('Calendar units and binary flags added:')
df[['date', 'year', 'month', 'day', 'day_of_week', 'week_of_year', 'quarter', 
    'is_weekend', 'is_month_start', 'is_month_end', 'is_payday']].head(5)


Calendar units and binary flags added:


,date,year,month,day,day_of_week,week_of_year,quarter,is_weekend,is_month_start,is_month_end,is_payday
0,2013-01-01,2013,1,1,1,1,1,0,1,0,0
1,2013-01-01,2013,1,1,1,1,1,0,1,0,0
2,2013-01-01,2013,1,1,1,1,1,0,1,0,0
3,2013-01-01,2013,1,1,1,1,1,0,1,0,0
4,2013-01-01,2013,1,1,1,1,1,0,1,0,0


### 4. Bước 1b — Binary Flags (Cuối tuần, Đầu/Cuối tháng, Ngày lương)

Ba binary flag nắm bắt các **thay đổi chế độ nhu cầu**:

- **`is_weekend`** — Thứ Bảy/Chủ Nhật thường có hành vi mua sắm khác hẳn ngày thường.
- **`is_month_start`** / **`is_month_end`** — nhu cầu thường tăng đột biến vào ngày đầu và cuối tháng.
- **`is_payday`** — tại Ecuador, lương được phát vào **ngày 15** và **ngày cuối tháng**. Đây là hai mốc thời gian tạo ra đợt tăng doanh số đáng tin cậy và là một trong những feature có tính dự đoán cao nhất trong bộ dữ liệu.

### 5. Tổng kết Temporal Features

Xác minh tất cả mười temporal feature đều có mặt và không có giá trị null bất thường.

In [17]:
TEMPORAL_FEATURE_NAMES = [
    'year', 'month', 'day', 'day_of_week', 'week_of_year', 'quarter',
    'is_weekend', 'is_month_start', 'is_month_end', 'is_payday',
]

print('Temporal feature check:')
summary = df[TEMPORAL_FEATURE_NAMES].agg(['min', 'max', lambda x: x.isna().sum()]).T
summary.columns = ['min', 'max', 'null_count']
print(summary)

Temporal feature check:
                 min   max  null_count
year            2013  2017           0
month              1    12           0
day                1    31           0
day_of_week        0     6           0
week_of_year       1    53           0
quarter            1     4           0
is_weekend         0     1           0
is_month_start     0     1           0
is_month_end       0     1           0
is_payday          0     1           0


---
## Phần 2 — Lag & Rolling Features

Lag features cho phép mô hình **trực tiếp truy cập các giá trị doanh số lịch sử**, đây là nguồn thông tin quan trọng nhất cho time-series forecasting.

### Chiến lược chống Data Leakage

Yêu cầu bắt buộc: không có feature nào tại thời điểm `t` được chứa thông tin từ thời điểm `t` hoặc sau đó. Ta đảm bảo điều này bằng hai biện pháp:

1. **Sort** dataframe theo `[store_nbr, family, date]` trước khi tính bất kỳ lag nào, để shift được áp dụng theo đúng thứ tự thời gian trong từng chuỗi.
2. **`shift(1)`** trên rolling statistics để rolling window cho dòng `t` chỉ bao gồm `[t-window-1, …, t-2, t-1]` — không bao giờ chứa `t`.

### 6. Bước 2a — Sắp xếp và Nhóm (Sort & Group)

Sắp xếp dataframe theo grouping key và date. Đây là bước **bắt buộc** trước khi gọi `.shift()` — nếu không sort, pandas sẽ tính lag theo thứ tự dòng tùy tiện, trộn lẫn các chuỗi thời gian khác nhau và gây ra data leakage.

In [18]:
GROUP_COLS  = ['store_nbr', 'family']
TARGET_COL  = 'sales'
DATE_COL    = 'date'

df = df.sort_values(GROUP_COLS + [DATE_COL]).reset_index(drop=True)

grouped = df.groupby(GROUP_COLS, sort=False)[TARGET_COL]

print(f'Sorted by: {GROUP_COLS + [DATE_COL]}')
print(f'Number of time series (store × family): {df.groupby(GROUP_COLS).ngroups}')
df[GROUP_COLS + [DATE_COL, TARGET_COL]].head(6)

Sorted by: ['store_nbr', 'family', 'date']
Number of time series (store × family): 1782


,store_nbr,family,date,sales
0,1,AUTOMOTIVE,2013-01-01,0.0
1,1,AUTOMOTIVE,2013-01-02,2.0
2,1,AUTOMOTIVE,2013-01-03,3.0
3,1,AUTOMOTIVE,2013-01-04,3.0
4,1,AUTOMOTIVE,2013-01-05,5.0
5,1,AUTOMOTIVE,2013-01-06,2.0


### 7. Bước 2b — Lag Ngắn hạn (1, 2, 3 ngày)

Lag ngắn hạn nắm bắt **động lượng tức thời** — doanh số đang tăng hay giảm trong vài ngày qua:
- `lag_1` — doanh số hôm qua (lag đơn lẻ có tính dự đoán cao nhất).
- `lag_2`, `lag_3` — hai và ba ngày trước, giúp mô hình phát hiện xu hướng ngắn hạn.

In [19]:
for lag in [1, 2, 3]:
    df[f'lag_{lag}'] = grouped.shift(lag)

print('Short-term lags added. NaN counts (expected at group boundaries):')
df[['lag_1', 'lag_2', 'lag_3']].isna().sum()

Short-term lags added. NaN counts (expected at group boundaries):


lag_1    1782
lag_2    3564
lag_3    5346
dtype: int64

### 8. Bước 2c — Lag theo Tuần (7 ngày)

`lag_7` thường là **lag feature quan trọng nhất** trong bán lẻ.  
Bằng cách căn chỉnh cùng thứ trong tuần, nó nắm bắt nhịp mua sắm hàng tuần mà không cần thêm tương tác riêng với `day_of_week`.

In [20]:
df['lag_7'] = grouped.shift(7)

print(f'lag_7 NaN count : {df["lag_7"].isna().sum()}')
print(f'lag_7 coverage  : {(1 - df["lag_7"].isna().mean()) * 100:.1f}%')

lag_7 NaN count : 12474
lag_7 coverage  : 99.6%


### 9. Bước 2d — Lag Dài hạn (14, 28, 364 ngày)

Các lag dài hơn nắm bắt những quy luật chuyển động chậm hơn:
- **`lag_14`** — nhịp hai tuần (phổ biến trong chu kỳ lương và bổ sung hàng).
- **`lag_28`** — xấp xỉ một tháng — làm mượt sự biến động theo tuần trong tháng.
- **`lag_364`** — cùng thứ trong tuần **một năm trước** (364 = 52 tuần × 7 ngày), căn chỉnh cả day-of-week lẫn vị trí lịch gần đúng để tạo tín hiệu seasonal mạnh.

In [21]:
for lag in [14, 28, 364]:
    df[f'lag_{lag}'] = grouped.shift(lag)

long_lag_cols = ['lag_14', 'lag_28', 'lag_364']
print('Longer lag NaN counts and coverage:')
for col in long_lag_cols:
    coverage = (1 - df[col].isna().mean()) * 100
    print(f'  {col:<10}  NaN={df[col].isna().sum():>6}   coverage={coverage:.1f}%')

Longer lag NaN counts and coverage:
  lag_14      NaN= 24948   coverage=99.2%
  lag_28      NaN= 49896   coverage=98.3%
  lag_364     NaN=648648   coverage=78.4%


### 10. Bước 2e — Rolling Statistics (cửa sổ 7, 14, 28 ngày)

Rolling statistics làm mượt nhiễu hàng ngày và cung cấp một **ước lượng mức nhu cầu cục bộ**:
- `rolling_mean_7`, `rolling_mean_14`, `rolling_mean_28` — trung bình doanh số trong N ngày gần nhất.
- `rolling_std_7` — độ lệch chuẩn rolling, đo lường **độ biến động nhu cầu** trong tuần qua.

**Chống Data Leakage**: chuỗi gốc được `shift(1)` trước khi rolling. Nghĩa là rolling window cho dòng `t` chỉ bao gồm `[t-window, …, t-1]` — không có ngày hiện tại `t`.  
`min_periods=1` đảm bảo các dòng gần đầu mỗi chuỗi thời gian vẫn nhận được ước lượng hợp lệ (cửa sổ chưa đầy) thay vì NaN.

In [22]:
# shift(1) guarantees the rolling window never includes the current observation
shifted = grouped.shift(1)

df['rolling_mean_7']  = shifted.transform(lambda x: x.rolling(7,  min_periods=1).mean())
df['rolling_mean_14'] = shifted.transform(lambda x: x.rolling(14, min_periods=1).mean())
df['rolling_mean_28'] = shifted.transform(lambda x: x.rolling(28, min_periods=1).mean())
df['rolling_std_7']   = shifted.transform(lambda x: x.rolling(7,  min_periods=2).std())

rolling_cols = ['rolling_mean_7', 'rolling_mean_14', 'rolling_mean_28', 'rolling_std_7']
print('Rolling feature summary:')
df[rolling_cols].describe().T[['mean', 'std', 'min', 'max']]

Rolling feature summary:


,mean,std,min,max
rolling_mean_7,357.704680,1047.254096,0.0,29937.285714
rolling_mean_14,357.711654,1041.093101,0.0,20097.642857
rolling_mean_28,357.717832,1034.296458,0.0,16118.321429
rolling_std_7,106.522145,354.699813,0.0,47275.358965


---
## Phần 3 — Pipeline Tổng hợp (Combined Pipeline)

Hai bước trên được đóng gói thành một hàm pipeline duy nhất `build_features` để tái sử dụng. Ở đây ta chạy thử pipeline đầy đủ trên **bản sao mới của dữ liệu thô** để xác nhận tính tái tạo.

### 11. Pipeline — `build_features` (Chạy toàn bộ)

Cell này gộp tất cả các bước từ Phần 1 và 2 vào một lần gọi hàm duy nhất. Dùng hàm này trong production pipeline khi muốn áp dụng toàn bộ feature trong một lần.

In [23]:
def add_temporal_features(df: pd.DataFrame, date_col: str = 'date') -> pd.DataFrame:
    """Extract calendar-based temporal features from the date column."""
    df = df.copy()
    df[date_col] = pd.to_datetime(df[date_col])
    dt = df[date_col].dt

    df['year']         = dt.year
    df['month']        = dt.month
    df['day']          = dt.day
    df['day_of_week']  = dt.dayofweek
    df['week_of_year'] = dt.isocalendar().week.astype(int)
    df['quarter']      = dt.quarter
    df['is_weekend']   = (dt.dayofweek >= 5).astype(int)
    df['is_month_start'] = dt.is_month_start.astype(int)
    df['is_month_end']   = dt.is_month_end.astype(int)
    df['is_payday']    = ((dt.day == 15) | dt.is_month_end).astype(int)
    return df


def add_lag_features(
    df: pd.DataFrame,
    target_col: str = 'sales',
    group_cols: list = None,
    date_col: str = 'date',
) -> pd.DataFrame:
    """Create lag and rolling features with anti-leakage guarantees."""
    if group_cols is None:
        group_cols = ['store_nbr', 'family']

    df = df.copy()
    df[date_col] = pd.to_datetime(df[date_col])
    df = df.sort_values(group_cols + [date_col]).reset_index(drop=True)

    grouped = df.groupby(group_cols, sort=False)[target_col]

    for lag in [1, 2, 3]:
        df[f'lag_{lag}'] = grouped.shift(lag)
    df['lag_7'] = grouped.shift(7)
    for lag in [14, 28, 364]:
        df[f'lag_{lag}'] = grouped.shift(lag)

    shifted = grouped.shift(1)
    df['rolling_mean_7']  = shifted.transform(lambda x: x.rolling(7,  min_periods=1).mean())
    df['rolling_mean_14'] = shifted.transform(lambda x: x.rolling(14, min_periods=1).mean())
    df['rolling_mean_28'] = shifted.transform(lambda x: x.rolling(28, min_periods=1).mean())
    df['rolling_std_7']   = shifted.transform(lambda x: x.rolling(7,  min_periods=2).std())
    return df


def build_features(
    df: pd.DataFrame,
    target_col: str = 'sales',
    group_cols: list = None,
    date_col: str = 'date',
) -> pd.DataFrame:
    """Full pipeline: temporal features → lag & rolling features."""
    df = add_temporal_features(df, date_col=date_col)
    df = add_lag_features(df, target_col=target_col, group_cols=group_cols, date_col=date_col)
    return df


# --- Run the full pipeline on real data ---
raw = pd.read_csv(rf'{BASE_PATH}\processed\train_cleaned.csv')
df_featured = build_features(raw)

print(f'Input shape  : {raw.shape}')
print(f'Output shape : {df_featured.shape}')
print(f'New columns  : {df_featured.shape[1] - raw.shape[1]}')

Input shape  : (3000888, 6)
Output shape : (3000888, 27)
New columns  : 21


---
## Phần 4 — Danh sách tên Feature (Feature Name Registry)

Ta định nghĩa ba danh sách tên feature chuẩn để dùng cho các bước downstream:
- **Chọn feature** khi train model (`X = df[ALL_FEATURE_NAMES]`).
- **Xác thực đầu ra** của pipeline.
- **Ghi chép** những cột nào được tạo ra từ feature engineering, cột nào là gốc.

### 12. Định nghĩa các Hằng số Tên Feature

Ba danh sách này được export để dùng trong các notebook và script train model downstream.

In [24]:
TEMPORAL_FEATURE_NAMES = [
    'year', 'month', 'day', 'day_of_week', 'week_of_year', 'quarter',
    'is_weekend', 'is_month_start', 'is_month_end', 'is_payday',
]

LAG_FEATURE_NAMES = [
    'lag_1', 'lag_2', 'lag_3',
    'lag_7',
    'lag_14', 'lag_28', 'lag_364',
    'rolling_mean_7', 'rolling_mean_14', 'rolling_mean_28',
    'rolling_std_7',
]

ALL_FEATURE_NAMES = TEMPORAL_FEATURE_NAMES + LAG_FEATURE_NAMES

print(f'Temporal features : {len(TEMPORAL_FEATURE_NAMES)}')
print(f'Lag features      : {len(LAG_FEATURE_NAMES)}')
print(f'Total features    : {len(ALL_FEATURE_NAMES)}')
print(f'\nAll features: {ALL_FEATURE_NAMES}')

Temporal features : 10
Lag features      : 11
Total features    : 21

All features: ['year', 'month', 'day', 'day_of_week', 'week_of_year', 'quarter', 'is_weekend', 'is_month_start', 'is_month_end', 'is_payday', 'lag_1', 'lag_2', 'lag_3', 'lag_7', 'lag_14', 'lag_28', 'lag_364', 'rolling_mean_7', 'rolling_mean_14', 'rolling_mean_28', 'rolling_std_7']


---
## Phần 5 — Xác thực & Kiểm tra Data Leakage

Trước khi đưa bất kỳ feature nào vào mô hình, cần xác minh:
1. **Tất cả feature mong đợi đều có mặt** trong dataframe đầu ra.
2. **Tỷ lệ NaN nằm trong giới hạn cho phép** (lag ở đầu mỗi chuỗi sẽ là NaN — điều này hoàn toàn đúng).
3. **Không có data leakage**: `lag_1` của dòng đầu tiên trong mỗi nhóm `(store_nbr, family)` phải là `NaN`, xác nhận rằng shift đã được áp dụng đúng trong từng group.

### 13. Smoke Test với Dữ liệu Giả (Mock Data)

Ta tạo một bộ dữ liệu tổng hợp nhỏ (2 stores × 60 ngày) và chạy toàn bộ pipeline để xác minh tính đúng đắn trong môi trường kiểm soát.

In [25]:
# --- Build mock data ---
dates = pd.date_range('2017-01-01', periods=60, freq='D')
mock = pd.DataFrame({
    'date'     : list(dates) * 2,
    'store_nbr': [1] * 60 + [2] * 60,
    'family'   : ['GROCERY I'] * 120,
    'sales'    : np.random.rand(120) * 1000,
})

# --- Run pipeline ---
result = build_features(mock)

print(f'Mock input shape  : {mock.shape}')
print(f'Mock output shape : {result.shape}')

Mock input shape  : (120, 4)
Mock output shape : (120, 25)


### 14. Kiểm tra sự Hiện diện Feature & Tỷ lệ NaN

Duyệt qua tất cả tên feature mong đợi và báo cáo:
- Feature có tồn tại không (`✓` / `✗ MISSING`).
- Tỷ lệ phần trăm NaN (kỳ vọng khác 0 cho các lag lớn như `lag_364` trong bộ dữ liệu 60 ngày).

### 15. Assertion chống Data Leakage

Dòng đầu tiên của mỗi nhóm `(store_nbr, family)` phải có `lag_1 = NaN` vì không có quan sát trước đó để tham chiếu. Nếu assertion này thất bại, nghĩa là shift đã được tính xuyên qua ranh giới group — đây là lỗi data leakage.

In [26]:
first_rows = result.groupby(['store_nbr', 'family']).first()

# diagnostic version that shows offending groups instead of failing silently
bad = first_rows[~first_rows['lag_1'].isna()]
if not bad.empty:
    print("Data leakage detected for the following groups:")
    display(bad[['lag_1']])
    raise AssertionError(
        "Data leakage detected! lag_1 of the first row in each group must be NaN."
    )

print('Anti-leakage check passed ✓')
print('lag_1 values for first row of each group:')
print(first_rows[['lag_1']])

Data leakage detected for the following groups:


,,lag_1
store_nbr,family,
1,GROCERY I,913.150323
2,GROCERY I,110.481825


AssertionError: Data leakage detected! lag_1 of the first row in each group must be NaN.

---
## 16. Lưu Kết quả (Save Output)

Lưu dataframe đã được bổ sung feature để các notebook modelling downstream có thể sử dụng.

In [ ]:
OUTPUT_PATH = rf'{BASE_PATH}\processed\train_temporal_features.csv'
df_featured.to_csv(OUTPUT_PATH, index=False)

print(f'Saved to : {OUTPUT_PATH}')
print(f'Shape    : {df_featured.shape}')
print(f'\nFinal columns:')
print(df_featured.columns.tolist())

Saved to : D:\Topic_13_Project\Topic_13_Retail_Store_Sales_Time_Series\data\processed\train_temporal_features.csv
Shape    : (3000888, 27)

Final columns:
['id', 'date', 'store_nbr', 'family', 'sales', 'onpromotion', 'year', 'month', 'day', 'day_of_week', 'week_of_year', 'quarter', 'is_weekend', 'is_month_start', 'is_month_end', 'is_payday', 'lag_1', 'lag_2', 'lag_3', 'lag_7', 'lag_14', 'lag_28', 'lag_364', 'rolling_mean_7', 'rolling_mean_14', 'rolling_mean_28', 'rolling_std_7']
